In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from io import StringIO
from scipy import stats

# ── Данные основного эксперимента (S1/S2/S3 + Baseline) ──────────────────────
RAW = """strategy,scenario,run,avgFrameMs,p95FrameMs,jankCount,totalFrames,dataKB,captureCount,durationMs
Baseline,SC01_FastScroll,1,17.826,19.564,424,426,0.000,0,9075
Baseline,SC01_FastScroll,2,18.022,19.759,427,427,0.000,0,9028
Baseline,SC01_FastScroll,3,18.024,19.834,426,427,0.000,0,9045
Baseline,SC01_FastScroll,4,18.083,20.004,428,428,0.000,0,9055
Baseline,SC01_FastScroll,5,18.072,19.805,424,426,0.000,0,9020
Baseline,SC02_SlowScroll,1,18.054,19.388,437,439,0.000,0,16382
Baseline,SC02_SlowScroll,2,17.880,19.365,439,441,0.000,0,15394
Baseline,SC02_SlowScroll,3,17.859,19.114,432,432,0.000,0,15423
Baseline,SC02_SlowScroll,4,17.847,19.114,438,438,0.000,0,15394
Baseline,SC02_SlowScroll,5,17.838,19.189,431,432,0.000,0,15427
Baseline,SC03_TapChats,1,18.463,19.575,131,132,0.000,0,10323
Baseline,SC03_TapChats,2,18.967,22.709,157,157,0.000,0,10323
Baseline,SC03_TapChats,3,18.009,20.117,157,157,0.000,0,10297
Baseline,SC03_TapChats,4,18.063,19.689,156,157,0.000,0,10266
Baseline,SC03_TapChats,5,17.801,19.514,154,156,0.000,0,10258
Baseline,SC05_SendMessages,1,18.926,39.392,27,30,0.000,0,4819
Baseline,SC05_SendMessages,2,18.563,33.695,52,52,0.000,0,4259
Baseline,SC05_SendMessages,3,18.120,19.622,52,52,0.000,0,4254
Baseline,SC05_SendMessages,4,18.133,21.568,53,53,0.000,0,4143
Baseline,SC05_SendMessages,5,18.814,33.827,52,52,0.000,0,4147
Baseline,SC06_NavigateTabs,1,18.564,20.656,158,160,0.000,0,8770
Baseline,SC06_NavigateTabs,2,19.193,22.860,158,160,0.000,0,8706
Baseline,SC06_NavigateTabs,3,18.953,22.028,160,160,0.000,0,8694
Baseline,SC06_NavigateTabs,4,18.576,22.730,158,160,0.000,0,8705
Baseline,SC06_NavigateTabs,5,18.623,22.367,155,160,0.000,0,8671
Baseline,SC07_RapidTaps,1,18.008,20.507,394,397,0.000,0,11416
Baseline,SC07_RapidTaps,2,18.375,22.210,420,423,0.000,0,11392
Baseline,SC07_RapidTaps,3,17.981,20.106,421,422,0.000,0,11383
Baseline,SC07_RapidTaps,4,18.410,21.277,422,422,0.000,0,11412
Baseline,SC07_RapidTaps,5,17.993,19.854,421,421,0.000,0,11381
Baseline,SC08_LongPress,1,18.986,23.387,228,228,0.000,0,11820
Baseline,SC08_LongPress,2,18.366,21.180,228,229,0.000,0,11766
Baseline,SC08_LongPress,3,18.307,21.117,229,229,0.000,0,11762
Baseline,SC08_LongPress,4,18.143,20.336,230,231,0.000,0,11775
Baseline,SC08_LongPress,5,18.419,21.559,229,229,0.000,0,11748
Baseline,SC09_MixedSession,1,19.426,23.396,235,237,0.000,0,12075
Baseline,SC09_MixedSession,2,19.188,22.632,317,318,0.000,0,11320
Baseline,SC09_MixedSession,3,19.289,22.680,282,284,0.000,0,11273
Baseline,SC09_MixedSession,4,19.163,22.815,283,283,0.000,0,11216
Baseline,SC09_MixedSession,5,19.183,22.687,287,287,0.000,0,11245
Baseline,SC10_TypeDelete,1,19.265,20.337,28,28,0.000,0,4580
Baseline,SC10_TypeDelete,2,20.450,51.027,51,51,0.000,0,4072
Baseline,SC10_TypeDelete,3,20.317,36.710,52,52,0.000,0,4034
Baseline,SC10_TypeDelete,4,21.053,35.187,53,53,0.000,0,4012
Baseline,SC10_TypeDelete,5,19.925,36.216,52,52,0.000,0,3948
S1_Bitmap,SC01_FastScroll,1,21.580,65.493,343,345,2890.938,16,9675
S1_Bitmap,SC01_FastScroll,2,19.644,19.978,359,360,2817.508,16,9371
S1_Bitmap,SC01_FastScroll,3,20.152,20.833,344,346,2697.530,16,9603
S1_Bitmap,SC01_FastScroll,4,19.978,20.390,355,357,2746.774,16,9487
S1_Bitmap,SC01_FastScroll,5,20.042,20.299,363,365,2719.616,16,9526
S2_ViewNode,SC01_FastScroll,1,18.099,19.805,429,430,18.492,18,9244
S2_ViewNode,SC01_FastScroll,2,18.160,20.013,428,432,18.492,18,9255
S2_ViewNode,SC01_FastScroll,3,18.040,19.735,426,429,18.492,18,9214
S2_ViewNode,SC01_FastScroll,4,18.099,19.741,425,429,18.492,18,9188
S2_ViewNode,SC01_FastScroll,5,18.012,19.904,427,433,18.492,18,9207
S3_ScanView,SC01_FastScroll,1,18.122,20.007,422,427,67.802,20,9984
S3_ScanView,SC01_FastScroll,2,18.056,19.777,428,432,62.437,20,9210
S3_ScanView,SC01_FastScroll,3,18.238,19.852,418,420,57.885,20,9475
S3_ScanView,SC01_FastScroll,4,18.056,19.692,427,427,73.033,20,9059
S3_ScanView,SC01_FastScroll,5,18.128,19.924,425,426,55.444,20,9066
S1_Bitmap,SC02_SlowScroll,1,20.694,39.452,414,414,5267.267,29,16933
S1_Bitmap,SC02_SlowScroll,2,19.547,19.429,415,417,5097.485,28,16251
S1_Bitmap,SC02_SlowScroll,3,19.794,19.565,412,413,5074.003,28,16251
S1_Bitmap,SC02_SlowScroll,4,19.587,19.885,412,414,5076.601,28,16223
S1_Bitmap,SC02_SlowScroll,5,19.693,19.857,409,411,5074.350,28,16224
S2_ViewNode,SC02_SlowScroll,1,17.926,19.148,429,432,32.875,32,16348
S2_ViewNode,SC02_SlowScroll,2,17.849,19.311,438,440,30.820,30,15436
S2_ViewNode,SC02_SlowScroll,3,17.917,19.197,435,438,30.820,30,15443
S2_ViewNode,SC02_SlowScroll,4,17.747,19.097,432,434,30.820,30,15427
S2_ViewNode,SC02_SlowScroll,5,17.891,19.061,433,433,30.820,30,15470
S3_ScanView,SC02_SlowScroll,1,17.759,18.996,431,433,26.526,6,15521
S3_ScanView,SC02_SlowScroll,2,17.738,19.052,428,431,30.287,6,15439
S3_ScanView,SC02_SlowScroll,3,17.850,19.257,433,433,31.057,6,15463
S3_ScanView,SC02_SlowScroll,4,17.737,19.371,425,429,24.452,6,15444
S3_ScanView,SC02_SlowScroll,5,17.835,19.128,432,434,32.358,6,15442
S1_Bitmap,SC03_TapChats,1,18.085,20.121,131,131,2836.569,18,10449
S1_Bitmap,SC03_TapChats,2,18.038,19.613,147,149,3338.455,18,10439
S1_Bitmap,SC03_TapChats,3,19.057,23.022,149,150,3338.455,18,10520
S1_Bitmap,SC03_TapChats,4,18.002,19.628,153,153,3338.455,18,10479
S1_Bitmap,SC03_TapChats,5,18.895,22.291,145,145,3152.985,17,10280
S2_ViewNode,SC03_TapChats,1,19.202,30.371,155,155,20.547,20,10333
S2_ViewNode,SC03_TapChats,2,18.687,24.010,155,155,20.547,20,10301
S2_ViewNode,SC03_TapChats,3,18.692,24.917,155,155,20.547,20,10293
S2_ViewNode,SC03_TapChats,4,18.562,23.333,155,155,20.547,20,10270
S2_ViewNode,SC03_TapChats,5,18.162,20.660,154,156,20.547,20,10268
S3_ScanView,SC03_TapChats,1,18.527,21.427,156,158,4.119,3,10341
S3_ScanView,SC03_TapChats,2,18.507,20.291,156,156,4.119,3,10290
S3_ScanView,SC03_TapChats,3,18.911,22.808,156,157,4.119,3,10307
S3_ScanView,SC03_TapChats,4,19.488,30.318,155,155,4.119,3,10312
S3_ScanView,SC03_TapChats,5,19.747,29.778,157,157,4.119,3,10333
S1_Bitmap,SC05_SendMessages,1,19.934,34.357,30,30,877.485,8,4999
S1_Bitmap,SC05_SendMessages,2,19.495,37.795,48,48,1298.288,7,4264
S1_Bitmap,SC05_SendMessages,3,20.039,33.935,45,46,1298.288,7,4146
S1_Bitmap,SC05_SendMessages,4,23.190,56.425,49,49,1298.288,7,4346
S1_Bitmap,SC05_SendMessages,5,19.559,33.204,44,44,1298.288,7,4189
S2_ViewNode,SC05_SendMessages,1,22.197,59.513,52,52,9.246,9,4553
S2_ViewNode,SC05_SendMessages,2,20.916,33.739,53,53,8.219,8,4246
S2_ViewNode,SC05_SendMessages,3,20.383,38.586,51,51,8.219,8,4231
S2_ViewNode,SC05_SendMessages,4,21.073,33.755,53,53,8.219,8,4241
S2_ViewNode,SC05_SendMessages,5,18.934,34.347,52,52,8.219,8,4228
S3_ScanView,SC05_SendMessages,1,20.529,36.294,52,52,1.502,1,4211
S3_ScanView,SC05_SendMessages,2,19.277,33.958,52,52,1.502,1,4214
S3_ScanView,SC05_SendMessages,3,23.285,41.729,53,53,1.502,1,4226
S3_ScanView,SC05_SendMessages,4,18.833,34.517,52,52,1.502,1,4216
S3_ScanView,SC05_SendMessages,5,18.909,34.899,51,51,1.502,1,4290
S1_Bitmap,SC06_NavigateTabs,1,22.643,53.640,137,143,1840.012,15,8917
S1_Bitmap,SC06_NavigateTabs,2,20.437,33.687,144,144,1947.172,15,8725
S1_Bitmap,SC06_NavigateTabs,3,19.488,20.697,145,149,1947.520,15,8744
S1_Bitmap,SC06_NavigateTabs,4,22.854,36.544,139,140,1783.301,14,8841
S1_Bitmap,SC06_NavigateTabs,5,19.904,21.639,146,149,1947.532,15,8794
S2_ViewNode,SC06_NavigateTabs,1,18.176,22.582,153,160,17.465,17,8721
S2_ViewNode,SC06_NavigateTabs,2,18.689,25.540,155,160,17.465,17,8705
S2_ViewNode,SC06_NavigateTabs,3,19.121,33.605,157,160,17.465,17,8711
S2_ViewNode,SC06_NavigateTabs,4,18.135,21.060,151,160,17.465,17,8743
S2_ViewNode,SC06_NavigateTabs,5,18.193,21.828,153,160,17.465,17,8705
S3_ScanView,SC06_NavigateTabs,1,18.469,23.215,153,160,0.191,0,8707
S3_ScanView,SC06_NavigateTabs,2,18.404,21.286,153,160,0.191,0,8710
S3_ScanView,SC06_NavigateTabs,3,18.570,22.468,155,160,0.191,0,8697
S3_ScanView,SC06_NavigateTabs,4,18.798,26.340,153,160,0.191,0,8693
S3_ScanView,SC06_NavigateTabs,5,18.624,25.435,152,160,0.191,0,8671
S1_Bitmap,SC07_RapidTaps,1,19.478,32.974,333,338,3409.651,19,12629
S1_Bitmap,SC07_RapidTaps,2,18.286,22.962,364,364,3338.455,18,12317
S1_Bitmap,SC07_RapidTaps,3,17.692,19.301,376,381,3523.925,19,12157
S1_Bitmap,SC07_RapidTaps,4,17.616,19.054,348,350,3338.455,18,12782
S1_Bitmap,SC07_RapidTaps,5,17.680,18.952,410,412,3709.395,20,11713
S2_ViewNode,SC07_RapidTaps,1,17.687,19.378,415,421,22.602,22,11346
S2_ViewNode,SC07_RapidTaps,2,17.619,19.750,415,423,22.602,22,11280
S2_ViewNode,SC07_RapidTaps,3,17.603,18.987,420,423,22.602,22,11288
S2_ViewNode,SC07_RapidTaps,4,17.372,18.539,419,424,22.602,22,11294
S2_ViewNode,SC07_RapidTaps,5,17.573,19.141,420,423,22.602,22,11267
S3_ScanView,SC07_RapidTaps,1,17.710,19.250,420,422,10.662,8,11387
S3_ScanView,SC07_RapidTaps,2,17.926,19.101,417,420,10.662,8,11392
S3_ScanView,SC07_RapidTaps,3,17.363,18.643,418,424,10.662,8,11267
S3_ScanView,SC07_RapidTaps,4,17.467,18.805,420,424,10.662,8,11256
S3_ScanView,SC07_RapidTaps,5,17.894,19.344,421,423,10.662,8,11255
S1_Bitmap,SC08_LongPress,1,26.042,59.349,192,193,3448.662,20,12169
S1_Bitmap,SC08_LongPress,2,18.794,26.669,216,216,3709.395,20,11822
S1_Bitmap,SC08_LongPress,3,18.387,20.596,219,219,3709.395,20,11883
S1_Bitmap,SC08_LongPress,4,18.804,23.049,216,217,3709.395,20,11871
S1_Bitmap,SC08_LongPress,5,18.386,23.651,216,216,3709.395,20,11865
S2_ViewNode,SC08_LongPress,1,18.961,26.922,229,232,23.629,23,11819
S2_ViewNode,SC08_LongPress,2,18.091,23.103,229,230,23.629,23,11721
S2_ViewNode,SC08_LongPress,3,18.924,23.765,230,232,23.629,23,11771
S2_ViewNode,SC08_LongPress,4,17.876,19.611,230,230,23.629,23,11776
S2_ViewNode,SC08_LongPress,5,19.281,33.641,223,225,23.629,23,11737
S3_ScanView,SC08_LongPress,1,18.617,23.847,228,230,5.455,4,11776
S3_ScanView,SC08_LongPress,2,18.375,22.471,231,231,5.455,4,11771
S3_ScanView,SC08_LongPress,3,17.710,20.206,227,231,5.455,4,11717
S3_ScanView,SC08_LongPress,4,17.640,19.424,230,231,5.455,4,11736
S3_ScanView,SC08_LongPress,5,19.433,29.801,227,227,5.455,4,11732
S1_Bitmap,SC09_MixedSession,1,22.036,38.977,260,265,3242.606,21,12667
S1_Bitmap,SC09_MixedSession,2,20.231,28.309,256,259,3362.063,20,11705
S1_Bitmap,SC09_MixedSession,3,19.104,20.609,255,258,3179.255,19,11379
S1_Bitmap,SC09_MixedSession,4,21.218,35.426,250,252,3287.967,19,11603
S1_Bitmap,SC09_MixedSession,5,21.770,22.992,248,248,2992.773,18,11552
S2_ViewNode,SC09_MixedSession,1,19.261,31.028,319,323,23.629,23,11700
S2_ViewNode,SC09_MixedSession,2,18.448,22.267,317,322,22.602,22,11490
S2_ViewNode,SC09_MixedSession,3,18.244,20.574,307,314,22.602,22,11242
S2_ViewNode,SC09_MixedSession,4,18.551,22.530,307,316,22.602,22,11451
S2_ViewNode,SC09_MixedSession,5,18.437,20.227,283,285,22.602,22,11417
S3_ScanView,SC09_MixedSession,1,18.066,20.093,277,281,16.763,6,11526
S3_ScanView,SC09_MixedSession,2,18.822,24.844,316,320,16.033,6,11436
S3_ScanView,SC09_MixedSession,3,18.846,21.051,310,314,13.900,6,11408
S3_ScanView,SC09_MixedSession,4,18.161,20.254,248,253,18.105,6,11308
S3_ScanView,SC09_MixedSession,5,18.619,22.530,282,286,18.133,6,11337
S1_Bitmap,SC10_TypeDelete,1,24.186,69.338,27,31,903.148,8,5022
S1_Bitmap,SC10_TypeDelete,2,18.730,33.698,49,49,1298.288,7,4020
S1_Bitmap,SC10_TypeDelete,3,20.745,56.379,44,46,1298.288,7,4140
S1_Bitmap,SC10_TypeDelete,4,18.898,34.006,49,49,1298.288,7,4074
S1_Bitmap,SC10_TypeDelete,5,19.096,33.796,49,49,1298.288,7,4076
S2_ViewNode,SC10_TypeDelete,1,19.191,34.039,52,52,7.191,7,3981
S2_ViewNode,SC10_TypeDelete,2,18.715,33.622,52,52,7.191,7,3983
S2_ViewNode,SC10_TypeDelete,3,18.664,19.952,52,52,7.191,7,3962
S2_ViewNode,SC10_TypeDelete,4,18.910,34.768,52,52,7.191,7,3999
S2_ViewNode,SC10_TypeDelete,5,21.608,38.968,53,53,8.219,8,4019
S3_ScanView,SC10_TypeDelete,1,18.929,35.403,51,52,1.502,1,4025
S3_ScanView,SC10_TypeDelete,2,18.641,34.535,52,52,1.502,1,3960
S3_ScanView,SC10_TypeDelete,3,18.911,33.800,52,52,1.502,1,3994
S3_ScanView,SC10_TypeDelete,4,19.089,33.742,52,52,1.502,1,3959
S3_ScanView,SC10_TypeDelete,5,18.840,34.505,52,52,1.502,1,4019
"""

df = pd.read_csv(StringIO(RAW.strip()))
df['jankPct'] = df['jankCount'] / df['totalFrames'] * 100

STRATEGY_ORDER  = ['Baseline', 'S1_Bitmap', 'S2_ViewNode', 'S3_ScanView']
STRATEGY_LABELS = ['Baseline', 'S1: Bitmap', 'S2: ViewNode', 'S3: ScanView']
COLORS = ['#757575', '#E53935', '#1E88E5', '#43A047']
JANK_THRESHOLD = 16.6
N = 5
T_CRIT = stats.t.ppf(0.975, df=N-1)

def ci95(s):
    return T_CRIT * s.std(ddof=1) / np.sqrt(len(s))

agg = df.groupby(['strategy', 'scenario']).agg(
    avgFrameMs_mean=('avgFrameMs','mean'), avgFrameMs_ci=('avgFrameMs', ci95),
    p95FrameMs_mean=('p95FrameMs','mean'), p95FrameMs_ci=('p95FrameMs', ci95),
    jankPct_mean   =('jankPct',   'mean'), jankPct_ci   =('jankPct',    ci95),
    dataKB_mean    =('dataKB',    'mean'), dataKB_ci    =('dataKB',     ci95),
).reset_index()

overall = df.groupby('strategy').agg(
    avgFrameMs_mean=('avgFrameMs','mean'), avgFrameMs_ci=('avgFrameMs', ci95),
    p95FrameMs_mean=('p95FrameMs','mean'), p95FrameMs_ci=('p95FrameMs', ci95),
    jankPct_mean   =('jankPct',   'mean'), jankPct_ci   =('jankPct',    ci95),
    dataKB_mean    =('dataKB',    'mean'), dataKB_ci    =('dataKB',     ci95),
).loc[STRATEGY_ORDER]

print('Сводная таблица (среднее по всем сценариям):')
print(overall[['avgFrameMs_mean','avgFrameMs_ci','p95FrameMs_mean','p95FrameMs_ci',
               'jankPct_mean','jankPct_ci']].round(2).to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 1 — Средняя длительность кадра (avgFrameMs)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(STRATEGY_LABELS))
bars = ax.bar(x, overall['avgFrameMs_mean'], yerr=overall['avgFrameMs_ci'],
              color=COLORS, capsize=6, width=0.55,
              error_kw=dict(elinewidth=1.5, capthick=1.5, ecolor='black'))
ax.axhline(JANK_THRESHOLD, color='grey', linestyle='--', linewidth=1,
           label=f'Бюджет кадра {JANK_THRESHOLD} мс (60 fps)')
for bar, val, ci in zip(bars, overall['avgFrameMs_mean'], overall['avgFrameMs_ci']):
    ax.text(bar.get_x()+bar.get_width()/2, val+ci+0.2,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(STRATEGY_LABELS)
ax.set_ylabel('Средняя длительность кадра, мс')
ax.set_title('M3 — Средняя длительность кадра\n(9 сценариев × 5 прогонов, планки — 95% ДИ)')
ax.legend(fontsize=9)
ax.set_ylim(0, (overall['avgFrameMs_mean']+overall['avgFrameMs_ci']).max()*1.25)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig1_avg_frame.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 2 — p95 длительность кадра
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(STRATEGY_LABELS))
bars = ax.bar(x, overall['p95FrameMs_mean'], yerr=overall['p95FrameMs_ci'],
              color=COLORS, capsize=6, width=0.55,
              error_kw=dict(elinewidth=1.5, capthick=1.5, ecolor='black'))
ax.axhline(JANK_THRESHOLD, color='grey', linestyle='--', linewidth=1,
           label=f'Бюджет кадра {JANK_THRESHOLD} мс')
for bar, val, ci in zip(bars, overall['p95FrameMs_mean'], overall['p95FrameMs_ci']):
    ax.text(bar.get_x()+bar.get_width()/2, val+ci+0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(STRATEGY_LABELS)
ax.set_ylabel('p95 длительность кадра, мс')
ax.set_title('M3 — 95-й перцентиль длительности кадра\n(планки — 95% ДИ)')
ax.legend(fontsize=9)
ax.set_ylim(0, (overall['p95FrameMs_mean']+overall['p95FrameMs_ci']).max()*1.25)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig2_p95_frame.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 3 — Доля джанк-кадров, %
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(STRATEGY_LABELS))
bars = ax.bar(x, overall['jankPct_mean'], yerr=overall['jankPct_ci'],
              color=COLORS, capsize=6, width=0.55,
              error_kw=dict(elinewidth=1.5, capthick=1.5, ecolor='black'))
for bar, val, ci in zip(bars, overall['jankPct_mean'], overall['jankPct_ci']):
    ax.text(bar.get_x()+bar.get_width()/2, val+ci+0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(STRATEGY_LABELS)
ax.set_ylabel('Доля кадров > 16.6 мс, %')
ax.set_title('M3 — Доля «джанк»-кадров по стратегиям\n(планки — 95% ДИ)')
ax.set_ylim(0, (overall['jankPct_mean']+overall['jankPct_ci']).max()*1.3)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig3_jank.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 4 — Объём данных (лог. шкала)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(STRATEGY_LABELS))
means = overall['dataKB_mean'].values
cis   = overall['dataKB_ci'].values

# Baseline имеет 0 KB — добавим маленькое eps для лог. шкалы
means_plot = np.where(means == 0, 0.01, means)
bars = ax.bar(x, means_plot, color=COLORS, width=0.55, log=True, alpha=0.9)
ax.errorbar(x[1:], means[1:], yerr=cis[1:], fmt='none',
            ecolor='black', elinewidth=1.5, capsize=6, capthick=1.5)

for i, (bar, val) in enumerate(zip(bars, means)):
    label = '0 KB' if val == 0 else f'{val:.0f} KB'
    ax.text(bar.get_x()+bar.get_width()/2, max(means_plot[i]*1.8, 0.02),
            label, ha='center', va='bottom', fontsize=9, fontweight='bold')

# Коэффициенты vs Baseline
for i in range(1, len(means)):
    if means[0] > 0:
        ratio = means[i] / means[0]
        ax.text(x[i], means_plot[i]/4,
                f'×{ratio:.0f}' if ratio >= 10 else f'×{ratio:.1f}',
                ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(STRATEGY_LABELS)
ax.set_ylabel('Объём данных, КБ (лог. шкала)')
ax.set_title('M4 — Объём записанных данных по стратегиям\n(лог. шкала, планки — 95% ДИ)')
ax.grid(axis='y', alpha=0.3, which='both')
plt.tight_layout(); plt.savefig('fig4_data_kb.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 5 — avgFrameMs по каждому сценарию (4 стратегии)
# ══════════════════════════════════════════════════════════════════════════════
scenarios = sorted(agg['scenario'].unique())
x = np.arange(len(scenarios))
n_strat = len(STRATEGY_ORDER)
width = 0.2
offsets = np.linspace(-(n_strat-1)/2*width, (n_strat-1)/2*width, n_strat)

fig, ax = plt.subplots(figsize=(14, 5))
for offset, strategy, color, label in zip(offsets, STRATEGY_ORDER, COLORS, STRATEGY_LABELS):
    d = agg[agg['strategy']==strategy].set_index('scenario').reindex(scenarios)
    ax.bar(x+offset, d['avgFrameMs_mean'], width, label=label, color=color, alpha=0.88,
           yerr=d['avgFrameMs_ci'], capsize=3, error_kw=dict(elinewidth=1, capthick=1, ecolor='black'))

ax.axhline(JANK_THRESHOLD, color='grey', linestyle='--', linewidth=1, label='Бюджет 16.6 мс')
ax.set_xticks(x); ax.set_xticklabels(scenarios, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Средняя длительность кадра, мс')
ax.set_title('M3 — avgFrameMs по сценариям и стратегиям (планки — 95% ДИ)')
ax.legend(fontsize=9, ncol=2)
ax.set_ylim(0, agg['avgFrameMs_mean'].max()*1.35)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('fig5_frame_per_scenario.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 6 — dataKB по каждому сценарию (лог. шкала)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 5))
for offset, strategy, color, label in zip(offsets, STRATEGY_ORDER, COLORS, STRATEGY_LABELS):
    d = agg[agg['strategy']==strategy].set_index('scenario').reindex(scenarios)
    vals = np.where(d['dataKB_mean'].fillna(0)==0, 0.01, d['dataKB_mean'].fillna(0))
    ax.bar(x+offset, vals, width, label=label, color=color, alpha=0.88, log=True)
    mask = d['dataKB_mean'].fillna(0) > 0
    if mask.any():
        ax.errorbar(x[mask]+offset, d['dataKB_mean'][mask], yerr=d['dataKB_ci'][mask],
                    fmt='none', ecolor='black', elinewidth=1, capsize=3, capthick=1)

ax.set_xticks(x); ax.set_xticklabels(scenarios, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Объём данных, КБ (лог. шкала)')
ax.set_title('M4 — Объём данных по сценариям и стратегиям (планки — 95% ДИ)')
ax.legend(fontsize=9, ncol=2)
ax.grid(axis='y', alpha=0.3, which='both')
plt.tight_layout(); plt.savefig('fig6_data_per_scenario.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Итоговая таблица
# ══════════════════════════════════════════════════════════════════════════════
def fmt(mean, ci): return f'{mean:.2f} ± {ci:.2f}'

table = pd.DataFrame({
    'Стратегия':    STRATEGY_LABELS,
    'avgFrame, мс': [fmt(m,c) for m,c in zip(overall['avgFrameMs_mean'], overall['avgFrameMs_ci'])],
    'p95Frame, мс': [fmt(m,c) for m,c in zip(overall['p95FrameMs_mean'], overall['p95FrameMs_ci'])],
    'Джанк, %':     [fmt(m,c) for m,c in zip(overall['jankPct_mean'],    overall['jankPct_ci'])],
    'Данные, КБ':   [fmt(m,c) for m,c in zip(overall['dataKB_mean'],     overall['dataKB_ci'])],
})
print('Сводная таблица (mean ± 95% ДИ, n=5 × 9 сценариев):')
print(table.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CPU ЭКСПЕРИМЕНТ — данные из Logcat (SC09 MixedSession, n=4)
# ══════════════════════════════════════════════════════════════════════════════
from io import StringIO

CPU_RESULT_RAW = """strategy,run,avgCpuPct,maxCpuPct,sampleCount
Baseline,1,14.84,53.00,23
Baseline,2,10.49,26.75,21
Baseline,3,14.15,39.64,22
Baseline,4,12.48,28.20,21
S1_Bitmap,1,22.37,39.00,21
S1_Bitmap,2,23.44,37.33,21
S1_Bitmap,3,23.64,52.88,22
S1_Bitmap,4,23.74,35.33,22
S2_ViewNode,1,12.87,27.94,21
S2_ViewNode,2,12.85,26.15,21
S2_ViewNode,3,13.78,32.93,21
S2_ViewNode,4,13.32,41.12,21
S3_ScanView,1,10.08,23.55,21
S3_ScanView,2,10.28,19.12,21
S3_ScanView,3,9.65,20.20,21
S3_ScanView,4,9.99,28.14,21"""

CPU_TIMELINE_RAW = """strategy,run,sampleIdx,cpuPct
Baseline,run1,0,0.00
Baseline,run1,1,0.80
Baseline,run1,2,38.05
Baseline,run1,3,26.40
Baseline,run1,4,29.28
Baseline,run1,5,1.20
Baseline,run1,6,7.39
Baseline,run1,7,24.95
Baseline,run1,8,53.00
Baseline,run1,9,3.99
Baseline,run1,10,23.80
Baseline,run1,11,5.77
Baseline,run1,12,13.20
Baseline,run1,13,10.80
Baseline,run1,14,20.76
Baseline,run1,15,20.60
Baseline,run1,16,10.98
Baseline,run1,17,7.19
Baseline,run1,18,18.96
Baseline,run1,19,0.20
Baseline,run1,20,8.58
Baseline,run1,21,8.40
Baseline,run1,22,6.96
Baseline,run2,0,0.00
Baseline,run2,1,0.60
Baseline,run2,2,19.16
Baseline,run2,3,19.12
Baseline,run2,4,14.17
Baseline,run2,5,11.38
Baseline,run2,6,18.36
Baseline,run2,7,2.19
Baseline,run2,8,0.00
Baseline,run2,9,10.98
Baseline,run2,10,17.96
Baseline,run2,11,4.60
Baseline,run2,12,26.75
Baseline,run2,13,16.17
Baseline,run2,14,14.40
Baseline,run2,15,0.00
Baseline,run2,16,17.17
Baseline,run2,17,0.00
Baseline,run2,18,8.18
Baseline,run2,19,5.17
Baseline,run2,20,13.86
Baseline,run3,0,0.00
Baseline,run3,1,0.39
Baseline,run3,2,28.54
Baseline,run3,3,7.20
Baseline,run3,4,27.54
Baseline,run3,5,24.95
Baseline,run3,6,11.58
Baseline,run3,7,16.73
Baseline,run3,8,0.00
Baseline,run3,9,9.98
Baseline,run3,10,4.20
Baseline,run3,11,20.83
Baseline,run3,12,19.36
Baseline,run3,13,39.64
Baseline,run3,14,28.80
Baseline,run3,15,20.96
Baseline,run3,16,6.99
Baseline,run3,17,16.60
Baseline,run3,18,0.00
Baseline,run3,19,9.78
Baseline,run3,20,6.60
Baseline,run3,21,10.60
Baseline,run4,0,0.00
Baseline,run4,1,0.40
Baseline,run4,2,27.74
Baseline,run4,3,23.80
Baseline,run4,4,26.35
Baseline,run4,5,12.77
Baseline,run4,6,19.36
Baseline,run4,7,0.20
Baseline,run4,8,5.19
Baseline,run4,9,8.58
Baseline,run4,10,11.00
Baseline,run4,11,6.37
Baseline,run4,12,22.95
Baseline,run4,13,16.33
Baseline,run4,14,10.98
Baseline,run4,15,9.58
Baseline,run4,16,18.60
Baseline,run4,17,0.00
Baseline,run4,18,8.18
Baseline,run4,19,5.39
Baseline,run4,20,28.20
S1_Bitmap,run1,0,0.00
S1_Bitmap,run1,1,15.74
S1_Bitmap,run1,2,29.40
S1_Bitmap,run1,3,27.94
S1_Bitmap,run1,4,38.00
S1_Bitmap,run1,5,24.00
S1_Bitmap,run1,6,19.56
S1_Bitmap,run1,7,28.74
S1_Bitmap,run1,8,2.00
S1_Bitmap,run1,9,25.35
S1_Bitmap,run1,10,22.40
S1_Bitmap,run1,11,16.53
S1_Bitmap,run1,12,39.00
S1_Bitmap,run1,13,31.08
S1_Bitmap,run1,14,27.60
S1_Bitmap,run1,15,23.75
S1_Bitmap,run1,16,27.60
S1_Bitmap,run1,17,7.97
S1_Bitmap,run1,18,14.34
S1_Bitmap,run1,19,25.20
S1_Bitmap,run1,20,23.55
S1_Bitmap,run2,0,0.20
S1_Bitmap,run2,1,16.53
S1_Bitmap,run2,2,32.27
S1_Bitmap,run2,3,29.00
S1_Bitmap,run2,4,25.90
S1_Bitmap,run2,5,24.95
S1_Bitmap,run2,6,19.00
S1_Bitmap,run2,7,29.80
S1_Bitmap,run2,8,3.39
S1_Bitmap,run2,9,29.28
S1_Bitmap,run2,10,21.60
S1_Bitmap,run2,11,18.36
S1_Bitmap,run2,12,37.33
S1_Bitmap,run2,13,29.60
S1_Bitmap,run2,14,34.73
S1_Bitmap,run2,15,27.40
S1_Bitmap,run2,16,8.40
S1_Bitmap,run2,17,27.29
S1_Bitmap,run2,18,17.17
S1_Bitmap,run2,19,26.10
S1_Bitmap,run2,20,33.86
S1_Bitmap,run2,21,0.00
S1_Bitmap,run3,0,0.00
S1_Bitmap,run3,1,17.37
S1_Bitmap,run3,2,29.34
S1_Bitmap,run3,3,28.80
S1_Bitmap,run3,4,25.15
S1_Bitmap,run3,5,24.80
S1_Bitmap,run3,6,18.16
S1_Bitmap,run3,7,27.89
S1_Bitmap,run3,8,4.20
S1_Bitmap,run3,9,25.70
S1_Bitmap,run3,10,37.45
S1_Bitmap,run3,11,20.76
S1_Bitmap,run3,12,52.88
S1_Bitmap,run3,13,34.13
S1_Bitmap,run3,14,25.00
S1_Bitmap,run3,15,26.60
S1_Bitmap,run3,16,8.96
S1_Bitmap,run3,17,30.20
S1_Bitmap,run3,18,15.97
S1_Bitmap,run3,19,22.20
S1_Bitmap,run3,20,20.76
S1_Bitmap,run3,21,23.80
S1_Bitmap,run4,0,0.00
S1_Bitmap,run4,1,16.47
S1_Bitmap,run4,2,32.07
S1_Bitmap,run4,3,30.34
S1_Bitmap,run4,4,30.60
S1_Bitmap,run4,5,27.00
S1_Bitmap,run4,6,21.16
S1_Bitmap,run4,7,31.54
S1_Bitmap,run4,8,0.40
S1_Bitmap,run4,9,28.69
S1_Bitmap,run4,10,24.95
S1_Bitmap,run4,11,17.96
S1_Bitmap,run4,12,35.33
S1_Bitmap,run4,13,34.73
S1_Bitmap,run4,14,32.53
S1_Bitmap,run4,15,25.95
S1_Bitmap,run4,16,13.80
S1_Bitmap,run4,17,21.76
S1_Bitmap,run4,18,16.93
S1_Bitmap,run4,19,25.35
S1_Bitmap,run4,20,33.33
S1_Bitmap,run4,21,21.36
S2_ViewNode,run1,0,0.00
S2_ViewNode,run1,1,1.80
S2_ViewNode,run1,2,25.35
S2_ViewNode,run1,3,23.35
S2_ViewNode,run1,4,17.96
S2_ViewNode,run1,5,11.98
S2_ViewNode,run1,6,10.80
S2_ViewNode,run1,7,12.15
S2_ViewNode,run1,8,0.20
S2_ViewNode,run1,9,18.16
S2_ViewNode,run1,10,13.15
S2_ViewNode,run1,11,6.59
S2_ViewNode,run1,12,27.94
S2_ViewNode,run1,13,25.75
S2_ViewNode,run1,14,23.00
S2_ViewNode,run1,15,0.20
S2_ViewNode,run1,16,18.36
S2_ViewNode,run1,17,0.00
S2_ViewNode,run1,18,7.58
S2_ViewNode,run1,19,7.95
S2_ViewNode,run1,20,18.00
S2_ViewNode,run2,0,0.00
S2_ViewNode,run2,1,1.00
S2_ViewNode,run2,2,24.75
S2_ViewNode,run2,3,26.15
S2_ViewNode,run2,4,22.36
S2_ViewNode,run2,5,0.80
S2_ViewNode,run2,6,19.21
S2_ViewNode,run2,7,0.00
S2_ViewNode,run2,8,4.19
S2_ViewNode,run2,9,6.79
S2_ViewNode,run2,10,20.16
S2_ViewNode,run2,11,13.17
S2_ViewNode,run2,12,25.35
S2_ViewNode,run2,13,22.55
S2_ViewNode,run2,14,17.40
S2_ViewNode,run2,15,9.56
S2_ViewNode,run2,16,18.56
S2_ViewNode,run2,17,0.20
S2_ViewNode,run2,18,13.37
S2_ViewNode,run2,19,4.19
S2_ViewNode,run2,20,20.12
S2_ViewNode,run3,0,0.00
S2_ViewNode,run3,1,1.00
S2_ViewNode,run3,2,24.75
S2_ViewNode,run3,3,25.80
S2_ViewNode,run3,4,22.95
S2_ViewNode,run3,5,12.60
S2_ViewNode,run3,6,18.33
S2_ViewNode,run3,7,2.60
S2_ViewNode,run3,8,0.20
S2_ViewNode,run3,9,16.37
S2_ViewNode,run3,10,21.91
S2_ViewNode,run3,11,4.99
S2_ViewNode,run3,12,32.93
S2_ViewNode,run3,13,25.10
S2_ViewNode,run3,14,21.16
S2_ViewNode,run3,15,0.20
S2_ViewNode,run3,16,22.71
S2_ViewNode,run3,17,0.20
S2_ViewNode,run3,18,5.39
S2_ViewNode,run3,19,8.78
S2_ViewNode,run3,20,21.36
S2_ViewNode,run4,0,0.00
S2_ViewNode,run4,1,0.40
S2_ViewNode,run4,2,23.15
S2_ViewNode,run4,3,24.35
S2_ViewNode,run4,4,18.60
S2_ViewNode,run4,5,0.00
S2_ViewNode,run4,6,24.85
S2_ViewNode,run4,7,0.20
S2_ViewNode,run4,8,7.57
S2_ViewNode,run4,9,11.98
S2_ViewNode,run4,10,25.15
S2_ViewNode,run4,11,10.96
S2_ViewNode,run4,12,41.12
S2_ViewNode,run4,13,7.14
S2_ViewNode,run4,14,22.80
S2_ViewNode,run4,15,9.36
S2_ViewNode,run4,16,19.40
S2_ViewNode,run4,17,0.40
S2_ViewNode,run4,18,0.20
S2_ViewNode,run4,19,12.13
S2_ViewNode,run4,20,19.92
S3_ScanView,run1,0,0.00
S3_ScanView,run1,1,0.20
S3_ScanView,run1,2,16.93
S3_ScanView,run1,3,16.17
S3_ScanView,run1,4,12.20
S3_ScanView,run1,5,11.40
S3_ScanView,run1,6,23.55
S3_ScanView,run1,7,0.60
S3_ScanView,run1,8,0.00
S3_ScanView,run1,9,12.57
S3_ScanView,run1,10,18.16
S3_ScanView,run1,11,4.59
S3_ScanView,run1,12,18.56
S3_ScanView,run1,13,19.36
S3_ScanView,run1,14,12.40
S3_ScanView,run1,15,0.00
S3_ScanView,run1,16,18.49
S3_ScanView,run1,17,0.00
S3_ScanView,run1,18,8.78
S3_ScanView,run1,19,4.78
S3_ScanView,run1,20,12.97
S3_ScanView,run2,0,0.00
S3_ScanView,run2,1,0.40
S3_ScanView,run2,2,15.77
S3_ScanView,run2,3,18.40
S3_ScanView,run2,4,17.93
S3_ScanView,run2,5,10.78
S3_ScanView,run2,6,19.12
S3_ScanView,run2,7,0.00
S3_ScanView,run2,8,3.99
S3_ScanView,run2,9,9.18
S3_ScanView,run2,10,12.00
S3_ScanView,run2,11,4.59
S3_ScanView,run2,12,16.60
S3_ScanView,run2,13,15.80
S3_ScanView,run2,14,17.96
S3_ScanView,run2,15,11.38
S3_ScanView,run2,16,16.77
S3_ScanView,run2,17,0.20
S3_ScanView,run2,18,7.57
S3_ScanView,run2,19,5.39
S3_ScanView,run2,20,12.13
S3_ScanView,run3,0,0.00
S3_ScanView,run3,1,0.00
S3_ScanView,run3,2,15.14
S3_ScanView,run3,3,16.17
S3_ScanView,run3,4,16.00
S3_ScanView,run3,5,12.57
S3_ScanView,run3,6,19.76
S3_ScanView,run3,7,0.00
S3_ScanView,run3,8,5.59
S3_ScanView,run3,9,7.37
S3_ScanView,run3,10,11.55
S3_ScanView,run3,11,6.99
S3_ScanView,run3,12,20.20
S3_ScanView,run3,13,15.37
S3_ScanView,run3,14,11.35
S3_ScanView,run3,15,0.20
S3_ScanView,run3,16,16.50
S3_ScanView,run3,17,0.20
S3_ScanView,run3,18,10.58
S3_ScanView,run3,19,2.59
S3_ScanView,run3,20,14.57
S3_ScanView,run4,0,0.00
S3_ScanView,run4,1,0.40
S3_ScanView,run4,2,17.00
S3_ScanView,run4,3,16.77
S3_ScanView,run4,4,14.80
S3_ScanView,run4,5,0.00
S3_ScanView,run4,6,19.52
S3_ScanView,run4,7,0.00
S3_ScanView,run4,8,5.58
S3_ScanView,run4,9,6.19
S3_ScanView,run4,10,12.97
S3_ScanView,run4,11,5.18
S3_ScanView,run4,12,17.56
S3_ScanView,run4,13,14.14
S3_ScanView,run4,14,10.60
S3_ScanView,run4,15,8.80
S3_ScanView,run4,16,19.32
S3_ScanView,run4,17,0.00
S3_ScanView,run4,18,8.78
S3_ScanView,run4,19,3.99
S3_ScanView,run4,20,28.14"""

df_res  = pd.read_csv(StringIO(CPU_RESULT_RAW))
df_time = pd.read_csv(StringIO(CPU_TIMELINE_RAW))

CPU_STRATEGIES = ['Baseline', 'S1_Bitmap', 'S2_ViewNode', 'S3_ScanView']
CPU_LABELS     = ['Baseline', 'S1: Bitmap', 'S2: ViewNode', 'S3: ScanView']
CPU_COLORS     = ['#757575', '#E53935', '#1E88E5', '#43A047']
T4 = stats.t.ppf(0.975, df=3)  # n=4

# Агрегация
cpu_sum = []
for s in CPU_STRATEGIES:
    d = df_res[df_res['strategy']==s]
    mu  = d['avgCpuPct'].mean()
    ci  = T4 * d['avgCpuPct'].std(ddof=1) / np.sqrt(len(d))
    mx  = d['maxCpuPct'].mean()
    cpu_sum.append({'strategy':s,'mu':mu,'ci':ci,'maxMean':mx})
cpu_df = pd.DataFrame(cpu_sum)

print('CPU сводная таблица (n=4, 95% ДИ):')
for _, r in cpu_df.iterrows():
    print(f"  {r.strategy:<12} avg={r.mu:.2f}±{r.ci:.2f}%  max={r.maxMean:.2f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 7 — CPU: столбцы avg + max (два подграфика рядом)
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('CPU-нагрузка стратегий (SC09 MixedSession, n=4, 95% ДИ)',
             fontsize=13, fontweight='bold')

x = np.arange(len(CPU_LABELS))

# Средний CPU
bars = ax1.bar(x, cpu_df['mu'], yerr=cpu_df['ci'],
               color=CPU_COLORS, edgecolor='black', linewidth=0.7,
               capsize=6, width=0.55,
               error_kw=dict(elinewidth=1.5, capthick=1.5, ecolor='black'))
for bar, row in zip(bars, cpu_df.itertuples()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+row.ci+0.3,
             f'{row.mu:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(CPU_LABELS, fontsize=10)
ax1.set_ylabel('Средний CPU процесса, %'); ax1.set_title('Средний CPU (avgCpuPct)')
ax1.set_ylim(0, (cpu_df['mu']+cpu_df['ci']).max()*1.3)
ax1.yaxis.grid(True, linestyle='--', alpha=0.5); ax1.set_axisbelow(True)

# Пиковый CPU
bars2 = ax2.bar(x, cpu_df['maxMean'],
                color=CPU_COLORS, edgecolor='black', linewidth=0.7, width=0.55, alpha=0.85)
for bar, row in zip(bars2, cpu_df.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{row.maxMean:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(CPU_LABELS, fontsize=10)
ax2.set_ylabel('Средний пиковый CPU, %'); ax2.set_title('Пиковый CPU (maxCpuPct)')
ax2.set_ylim(0, cpu_df['maxMean'].max()*1.3)
ax2.yaxis.grid(True, linestyle='--', alpha=0.5); ax2.set_axisbelow(True)

plt.tight_layout(); plt.savefig('fig7_cpu_bar.png', dpi=150); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# График 8 — CPU временной ряд (ломаные, усред. по 4 прогонам, ДИ)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_title('CPU% во времени (среднее по 4 прогонам, 95% ДИ)', fontsize=13, fontweight='bold')

for strat, color, label in zip(CPU_STRATEGIES, CPU_COLORS, CPU_LABELS):
    sub = df_time[df_time['strategy']==strat]
    grouped = sub.groupby('sampleIdx')['cpuPct']
    mu  = grouped.mean()
    sem = grouped.sem().fillna(0)
    ci  = T4 * sem
    t   = mu.index * 0.5  # sampleIdx × 500 мс
    ax.plot(t, mu, color=color, linewidth=2.2, label=label)
    ax.fill_between(t, (mu-ci).clip(0), mu+ci, color=color, alpha=0.15)

ax.set_xlabel('Время, с', fontsize=11)
ax.set_ylabel('CPU процесса, %', fontsize=11)
ax.set_ylim(bottom=0)
ax.legend(fontsize=10, framealpha=0.9)
ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig('fig8_cpu_timeline.png', dpi=150); plt.show()